In [ ]:
import requests
import pandas as pd
import time
import uuid
from pyspark.sql import functions as F

base_url = "https://hubeau.eaufrance.fr/api/v1/qualite_eau_potable/resultats_dis"
code_departement = "59"
size = 1000
request_timeout = 30
max_retries = 5
retry_backoff_seconds = 3
state_table = "main.bronze.qualite_eau_region_ingestion_state"
target_table = "main.bronze.qualite_eau_region"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {state_table} (
  run_id STRING,
  code_departement STRING,
  page_size INT,
  last_success_page INT,
  status STRING,
  last_error STRING,
  updated_at TIMESTAMP
)
USING DELTA
""")

state_df = spark.sql(f"""
SELECT last_success_page
FROM {state_table}
WHERE code_departement = '{code_departement}'
ORDER BY updated_at DESC
LIMIT 1
""")

last_success_page = state_df.collect()[0]["last_success_page"] if state_df.count() > 0 else 0
start_page = int(last_success_page) + 1
run_id = str(uuid.uuid4())

print(f"Resume from page={start_page} (last_success_page={last_success_page})")

session = requests.Session()
last_written_df = None


def write_state(page: int, status: str, last_error: str = None) -> None:
    payload = [(
        run_id,
        code_departement,
        int(size),
        int(page),
        status,
        last_error,
    )]
    (spark.createDataFrame(
        payload,
        [
            "run_id",
            "code_departement",
            "page_size",
            "last_success_page",
            "status",
            "last_error",
        ],
    )
    .withColumn("updated_at", F.current_timestamp())
    .write.mode("append")
    .saveAsTable(state_table))


page = start_page

while True:
    url = f"{base_url}?code_departement={code_departement}&page={page}&size={size}"

    response = None
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(url, timeout=request_timeout)
            response.raise_for_status()
            break
        except requests.exceptions.RequestException as err:
            if attempt == max_retries:
                write_state(page - 1, "FAILED", str(err))
                raise
            sleep_s = retry_backoff_seconds * attempt
            print(f"Retry {attempt}/{max_retries} failed on page={page}: {err}")
            print(f"Sleeping {sleep_s}s before retry...")
            time.sleep(sleep_s)

    json_data = response.json()
    data = json_data.get("data", [])

    if not data:
        write_state(page - 1, "SUCCESS", None)
        print("No more data. Ingestion completed.")
        break

    pdf = pd.DataFrame(data)
    pdf = pdf.where(pd.notnull(pdf), None)
    df = spark.createDataFrame(pdf)

    df = df.withColumn("source_url", F.lit(base_url)) \
           .withColumn("ingestion_type", F.lit("api")) \
           .withColumn("ingested_at", F.current_timestamp()) \
           .withColumn("ingestion_run_id", F.lit(run_id)) \
           .withColumn("ingestion_page", F.lit(page))

    hash_cols = [c for c in df.columns if c not in ["ingested_at"]]
    df = df.withColumn("hash", F.sha2(F.to_json(F.struct(*hash_cols)), 256))
    df = df.withColumn("annee", F.year(F.to_timestamp("date_prelevement")))

    (df.write.format("delta")
        .mode("append")
        .partitionBy("annee")
        .saveAsTable(target_table))

    write_state(page, "IN_PROGRESS", None)
    print(f"Page {page} appended: rows={len(data)}")
    last_written_df = df

    if len(data) < size:
        write_state(page, "SUCCESS", None)
        print("Last partial page reached. Ingestion completed.")
        break

    page += 1

if last_written_df is not None:
    display(last_written_df)
else:
    print("No rows appended in this run.")

In [ ]:
%pip install great_expectations

In [ ]:
%sql
DESCRIBE TABLE main.bronze.qualite_eau_region;

In [ ]:
%sql
SELECT * FROM main.bronze.qualite_eau_region LIMIT 10;

In [ ]:
%sql
SELECT *
FROM main.bronze.qualite_eau_region_ingestion_state
ORDER BY updated_at DESC
LIMIT 20;